# V11 - Aplicacoes consumidoras: do dado a decisao

**Objetivo:** implementar o contrato de uma aplicacao consumidora sobre dados de teste: filtro, resultado limitado, estado vazio e mapeamento da resposta para a tela (ranking/KPI), com grafico.

**Padrao de consumo:** a app le pela `view`, aplica um filtro de negocio, agrega e apresenta uma decisao.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v11',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Preparar dados de teste
Cria um contrato com `name`, `category` e `score` e popula alguns nodes.

In [ ]:
from uuid import uuid4
import pandas as pd
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, Float64, MappedPropertyApply, NodeApply, NodeId,
    NodeOrEdgeData, SpaceApply, Text, ViewApply, ViewId,
)

run = uuid4().hex[:8]
sp = f'sp_ur_training_v11_{run}'
container_id = ContainerId(sp, 'Asset')
view_id = ViewId(sp, 'Asset', 'v1')

client.data_modeling.spaces.apply(SpaceApply(space=sp, name='UR training - V11'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': sp, 'externalId': container_id.external_id,
    'properties': {
        'name': {'type': Text().dump(), 'nullable': False},
        'category': {'type': Text().dump(), 'nullable': True},
        'score': {'type': Float64().dump(), 'nullable': True},
    },
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=sp, external_id=view_id.external_id, version=view_id.version,
    properties={
        'name': MappedPropertyApply(container=container_id, container_property_identifier='name'),
        'category': MappedPropertyApply(container=container_id, container_property_identifier='category'),
        'score': MappedPropertyApply(container=container_id, container_property_identifier='score'),
    },
))
sample = [
    ('a1', 'Bomba A', 'rotativo', 0.92),
    ('a2', 'Bomba B', 'rotativo', 0.61),
    ('a3', 'Valvula C', 'estatico', 0.45),
    ('a4', 'Valvula D', 'estatico', 0.78),
    ('a5', 'Motor E', 'rotativo', 0.33),
]
client.data_modeling.instances.apply(nodes=[
    NodeApply(space=sp, external_id=xid,
              sources=[NodeOrEdgeData(source=view_id, properties={'name': nm, 'category': cat, 'score': sc})])
    for xid, nm, cat, sc in sample
])
print('dados de teste criados:', len(sample), 'nodes em', sp)

## 2. Ler pelo contrato e montar o DataFrame da aplicacao

In [ ]:
nodes = client.data_modeling.instances.list(instance_type='node', sources=view_id, space=sp, limit=100)
df = nodes.to_pandas()
cols = [c for c in ['name', 'category', 'score'] if c in df.columns]
app_df = df[cols].copy()
app_df

## 3. Filtro de negocio + estado vazio
A app so decide sobre itens com `score >= 0.5`; trata o caso vazio.

In [ ]:
threshold = 0.5
filtered = app_df[app_df['score'] >= threshold].sort_values('score', ascending=False)
if filtered.empty:
    print('Estado vazio: nenhum item atende ao criterio; a tela mostraria uma mensagem amigavel.')
else:
    print(f'{len(filtered)} item(ns) acima de {threshold}.')
filtered

## 4. Decisao: ranking/KPI por categoria + grafico

In [ ]:
import matplotlib.pyplot as plt

kpi = app_df.groupby('category')['score'].mean().sort_values(ascending=False)
print('score medio por categoria:')
print(kpi)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(kpi.index.astype(str), kpi.values)
ax.set_title('Decisao: score medio por categoria')
ax.set_ylabel('score medio')
ax.set_ylim(0, 1)
for i, v in enumerate(kpi.values):
    ax.text(i, v, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Mini-exercicio
- Troque a metrica de decisao para `count` por categoria (quantos itens) e refaca o grafico.
- Mude o `threshold` para 0.9 e confirme o tratamento de estado vazio.

## 5. Limpeza idempotente

In [ ]:
assert sp.startswith('sp_ur_training_v11_')
client.data_modeling.instances.delete(nodes=[NodeId(sp, xid) for xid, *_ in sample])
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
client.data_modeling.spaces.delete(sp)
print('space_still_exists:', client.data_modeling.spaces.retrieve(sp) is not None)